**Install the required libraries**

In [ ]:
!pip install -q google-genai supabase faster-whisper gradio gTTS

**LOAD VYNZORA CREDENTIALS**

In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

SUPABASE_URL = userdata.get("SUPABASE_URL")

SUPABASE_KEY = userdata.get("SUPABASE_ANON_KEY")

SUPABASE_EMAIL = userdata.get("SUPABASE_EMAIL")

SUPABASE_PASSWORD = userdata.get("SUPABASE_PASSWORD")

**CONNECT SUPABASE**

In [ ]:
from supabase import create_client
from google.colab import userdata

url = userdata.get("SUPABASE_URL")
key = userdata.get("SUPABASE_ANON_KEY")

supabase = create_client(url, key)

print("Vynzora Supabase client connected!")

**CONNECT GEMINI**

In [ ]:
from google import genai

gemini = genai.Client(
    api_key=GEMINI_API_KEY
)

print("Gemini connected successfully")

In [ ]:
# Authenticate with your dedicated Supabase user
email = userdata.get("SUPABASE_EMAIL")
password = userdata.get("SUPABASE_PASSWORD")

supabase.auth.sign_in_with_password({
    "email": email,
    "password": password
})

print("Connected to Vynzora database")

In [ ]:
ALLOWED_TABLES = [
    "contacts",
    "crm_leads",
    "crm_activities",
    "notifications"
]

DATE_COLUMNS = {
    "contacts": "created_at",
    "crm_leads": "created_at",
    "crm_activities": "activity_date",
    "notifications": "created_at"
}


def read_database(table, limit=10):

    if table not in ALLOWED_TABLES:
        return "Table not allowed"

    try:

        date_column = DATE_COLUMNS[table]

        response = (
            supabase
            .table(table)
            .select("*")
            .order(date_column, desc=True)
            .limit(limit)
            .execute()
        )

        return response.data

    except Exception as e:

        return f"Database error: {str(e)}"

In [ ]:
result = read_database(
    "contacts",
    5
)

print(result)

**Customer Search Function**

In [ ]:
def search_customer(name):

    try:

        response = (
            supabase
            .table("contacts")
            .select("*")
            .ilike("full_name", f"%{name}%")
            .limit(10)
            .execute()
        )

        return response.data

    except Exception as e:

        return str(e)

In [ ]:
result = search_customer("Amara Okafor")

print(result)

**Retrieve complete customer information**

In [ ]:
def get_customer_complete_details(name):

    # 1. Search customer
    contacts = search_customer(name)

    if not contacts:
        return "Customer not found"

    if len(contacts) > 1:
        return {
            "message": "Multiple customers found",
            "customers": contacts
        }

    contact = contacts[0]

    contact_id = contact["id"]

    # 2. Find CRM lead
    leads = (
        supabase
        .table("crm_leads")
        .select("*")
        .eq("contact_id", contact_id)
        .execute()
    ).data

    # 3. Retrieve activities
    activities = []

    for lead in leads:

        lead_id = lead["id"]

        activity = (
            supabase
            .table("crm_activities")
            .select("*")
            .eq("lead_id", lead_id)
            .execute()
        ).data

        activities.extend(activity)

    return {
        "contact_details": contact,
        "lead_details": leads,
        "activities": activities
    }

In [ ]:
result = get_customer_complete_details("")

print(result)

**Insert customer into contacts**

In [ ]:
def create_customer(full_name, email, phone_number, company_name, service_required, message):

    data = {
        "full_name": full_name,
        "email": email,
        "phone_number": phone_number,
        "company_name": company_name,
        "service_required": service_required,
        "message": message
    }

    response = supabase.table("contacts").insert(data).execute()

    if response.data:
        print(f"{full_name} has been successfully added to contacts!")
        print("Customer Details:")
        print(response.data[0])
    else:
        print("Customer could not be added.")

    return response.data

In [ ]:
create_customer(
    full_name="Manoj",
    email="manoj@gmail.com",
    phone_number="9876543210",
    company_name="ABC Technologies",
    service_required="Digital Marketing",
    message="Interested in digital marketing services"
)

**Update an Existing CRM Lead.**

In [ ]:
def update_lead(customer_name, status, notes):

    # Find customer
    customer = supabase.table("contacts") \
        .select("id, full_name") \
        .ilike("full_name", customer_name) \
        .execute()

    if not customer.data:
        print("Customer not found!")
        return

    if len(customer.data) > 1:
        print("Multiple customers found. Please identify the correct customer.")
        return

    contact_id = customer.data[0]["id"]

    # Find corresponding lead
    lead = supabase.table("crm_leads") \
        .select("*") \
        .eq("contact_id", contact_id) \
        .execute()

    if not lead.data:
        print("No lead found for this customer!")
        return

    if len(lead.data) > 1:
        print("Multiple leads found. Please identify the correct lead.")
        return

    lead_id = lead.data[0]["id"]

    # Update lead
    response = supabase.table("crm_leads") \
        .update({
            "lead_status": status,
            "notes": notes
        }) \
        .eq("id", lead_id) \
        .execute()

    print(f"{customer_name}'s lead updated successfully!")
    print(response.data)

    return response.data

In [ ]:
update_lead(
    customer_name="Manoj",
    status="Contacted",
    notes="Called Manoj regarding Digital Marketing services."
)

**Create the delete function**

In [ ]:
def delete_lead(customer_name):

    # Find customer
    customer = supabase.table("contacts") \
        .select("id, full_name") \
        .ilike("full_name", customer_name) \
        .execute()

    if not customer.data:
        print("Customer not found!")
        return

    if len(customer.data) > 1:
        print("Multiple customers found. Please identify the correct customer.")
        return

    contact_id = customer.data[0]["id"]

    # Find lead
    lead = supabase.table("crm_leads") \
        .select("id") \
        .eq("contact_id", contact_id) \
        .execute()

    if not lead.data:
        print("No lead found!")
        return

    if len(lead.data) > 1:
        print("Multiple leads found. Please identify the correct lead.")
        return

    lead_id = lead.data[0]["id"]

    # Delete lead
    response = supabase.table("crm_leads") \
        .delete() \
        .eq("id", lead_id) \
        .execute()

    if response.data:
        print(f"{customer_name}'s CRM lead deleted successfully!")
    else:
        print("Lead deletion could not be confirmed.")

    return response.data

In [ ]:
delete_lead("Manoj")

**CRM Analytics**

Get total enquiries


In [ ]:
def get_total_enquiries():

    response = supabase.table("contacts") \
        .select("id") \
        .execute()

    total = len(response.data)

    print("Total Enquiries:", total)

    return total

In [ ]:
get_total_enquiries()

Get total CRM leads

In [ ]:
def get_total_leads():

    response = supabase.table("crm_leads") \
        .select("id") \
        .execute()

    total = len(response.data)

    print("Total CRM Leads:", total)

    return total

In [ ]:
get_total_leads()

Count leads by status

In [ ]:
def get_leads_by_status():

    response = supabase.table("crm_leads") \
        .select("lead_status") \
        .execute()

    status_count = {}

    for lead in response.data:
        status = lead["lead_status"]

        status_count[status] = status_count.get(status, 0) + 1

    print("CRM Lead Summary:")

    for status, count in status_count.items():
        print(status, ":", count)

    return status_count

In [ ]:
get_leads_by_status()

Get the latest five enquiries

In [ ]:
def get_latest_enquiries():

    response = supabase.table("contacts") \
        .select("*") \
        .order("created_at", desc=True) \
        .limit(5) \
        .execute()

    print("Latest Enquiries:")

    for customer in response.data:
        print(
            customer["full_name"],
            "-",
            customer["service_required"],
            "-",
            customer["created_at"]
        )

    return response.data

In [ ]:
get_latest_enquiries()

Combine all analytics

In [ ]:
def get_crm_analytics():

    total_enquiries = get_total_enquiries()

    total_leads = get_total_leads()

    lead_status = get_leads_by_status()

    print("\n========== CRM ANALYTICS ==========")
    print("Total Enquiries:", total_enquiries)
    print("Total Leads:", total_leads)
    print("Lead Status:", lead_status)

    return {
        "total_enquiries": total_enquiries,
        "total_leads": total_leads,
        "lead_status": lead_status
    }

In [ ]:
get_crm_analytics()

**Notifications Management.**

Retrieve unread notifications

In [ ]:
def get_unread_notifications():

    response = supabase.table("notifications") \
        .select("*") \
        .eq("is_read", False) \
        .execute()

    print("Unread Notifications:", len(response.data))

    for notification in response.data:
        print(notification["title"])
        print(notification["description"])
        print("Priority:", notification["priority"])
        print("----------------------")

    return response.data

In [ ]:
get_unread_notifications()

Retrieve today's notifications

In [ ]:
from datetime import date

def get_today_notifications():

    today = date.today().isoformat()

    response = supabase.table("notifications") \
        .select("*") \
        .eq("scheduled_date", today) \
        .execute()

    print("Today's Notifications:", len(response.data))

    for notification in response.data:
        print(notification["title"])
        print(notification["scheduled_time"])
        print("Priority:", notification["priority"])
        print("----------------------")

    return response.data

In [ ]:
get_today_notifications()

Retrieve high-priority notifications

In [ ]:
def get_high_priority_notifications():

    response = supabase.table("notifications") \
        .select("*") \
        .eq("priority", "High") \
        .execute()

    print("High Priority Notifications:", len(response.data))

    for notification in response.data:
        print(notification["title"])
        print(notification["description"])
        print("----------------------")

    return response.data

In [ ]:
get_high_priority_notifications()

Retrieve scheduled notifications

In [ ]:
def get_scheduled_notifications():

    response = supabase.table("notifications") \
        .select("*") \
        .eq("status", "Scheduled") \
        .execute()

    print("Scheduled Notifications:", len(response.data))

    for notification in response.data:
        print(notification["title"])
        print("Date:", notification["scheduled_date"])
        print("Time:", notification["scheduled_time"])
        print("----------------------")

    return response.data

In [ ]:
get_scheduled_notifications()

Mark a notification as read

In [ ]:
def mark_notification_as_read(title):

    response = supabase.table("notifications") \
        .select("*") \
        .ilike("title", f"%{title}%") \
        .execute()

    if not response.data:
        print("Notification not found!")
        return

    if len(response.data) > 1:
        print("Multiple notifications found. Please specify the title.")
        return

    notification_id = response.data[0]["id"]

    result = supabase.table("notifications") \
        .update({"is_read": True}) \
        .eq("id", notification_id) \
        .execute()

    if result.data:
        print("Notification marked as read successfully!")
        print(result.data[0])

    return result.data

In [ ]:
mark_notification_as_read("Follow-up Reminder")

**Integrate Gemini AI with the Python database functions.**

Install Gemini library

In [ ]:
!pip install -q google-genai

In [ ]:
response = gemini.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Hello! Introduce yourself as a Vynzora AI Business Assistant."
)

print(response.text)

Connect Python functions with Gemini

let's connect only the read and analytics functions.

In [ ]:
crm_tools = [
    get_total_enquiries,
    get_total_leads,
    get_leads_by_status,
    get_latest_enquiries,
    get_crm_analytics,
    get_unread_notifications,
    get_today_notifications,
    get_high_priority_notifications,
    get_scheduled_notifications
]

Create the AI agent

In [ ]:
system_instruction = """
You are Vynzora AI Business Assistant.

You help users manage and analyse the Vynzora CRM database.

You can:
- Retrieve customer enquiries.
- Analyse CRM leads.
- Count leads by status.
- Retrieve notifications.
- Generate CRM summaries.

Rules:
1. Use Python tools to retrieve actual database information.
2. Never invent customer records or analytics.
3. Answer using actual results returned by the tools.
4. Understand Malayalam, English and Hindi.
5. Always respond in the same language used by the user.
6. Keep customer names and database information accurate.
7. Explain results in simple language.
8. If the user asks a database question, use the relevant tools.
"""

In [ ]:
from google.genai import types

MODEL_NAME = "gemini-3.5-flash-lite"

chat = gemini.chats.create(
    model=MODEL_NAME,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        tools=crm_tools
    )
)

print("Vynzora AI Agent created successfully!")

In [ ]:
response = chat.send_message(
    "How many total enquiries do we have?"
)

print(response.text)

In [ ]:
response = chat.send_message(
    "നമുക്ക് എത്ര പുതിയ enquiries ഉണ്ട്?"
)

print(response.text)

Create the multilingual voice assistant

In [ ]:
!pip install -q gTTS

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display

In [ ]:
from google.colab import files

uploaded = files.upload()

audio_path = next(iter(uploaded))

print("Audio uploaded:", audio_path)

In [ ]:
!pip install -q sarvamai

In [ ]:
from sarvamai import SarvamAI
from google.colab import userdata

sarvam_api_key = userdata.get("SARVAM_API_KEY")

client = SarvamAI(
    api_subscription_key=sarvam_api_key
)

print("Sarvam AI connected successfully!")

In [ ]:
def transcribe_audio(audio_path):

    with open(audio_path, "rb") as audio_file:

        response = client.speech_to_text.transcribe(
            file=audio_file,
            model="saaras:v3",
            mode="transcribe"
        )

    transcript = response.transcript

    detected_language = response.language_code

    print("Detected Language:", detected_language)
    print("Recognized Text:", transcript)

    return transcript, detected_language

In [ ]:
transcript, detected_language = transcribe_audio(audio_path)

In [ ]:
!nvidia-smi

In [ ]:
def voice_assistant(audio_path):

    # 1. Speech-to-Text using Sarvam AI
    transcript, detected_language = transcribe_audio(audio_path)

    print("\nUser Question:")
    print(transcript)

    print("\nDetected Language:", detected_language)

    # 2. Convert Sarvam language code to gTTS language code
    language_mapping = {
        "ml-IN": "ml",
        "en-IN": "en",
        "hi-IN": "hi"
    }

    tts_language = language_mapping.get(
        detected_language,
        "en"
    )

    # 3. Send the question to Gemini
    prompt = f"""
    The user's language is {detected_language}.

    Understand the user's question and respond in the same language.

    Do not translate the answer into another language.

    Use the available Python database tools
    whenever the question requires CRM information.

    Never invent database information.

    User question:
    {transcript}
    """

    response = chat.send_message(prompt)

    ai_response = response.text

    print("\nVynzora AI Response:")
    print(ai_response)

    # 4. Convert Gemini response into speech
    tts = gTTS(
        text=ai_response,
        lang=tts_language,
        slow=False
    )

    audio_file = "vynzora_voice_response.mp3"

    tts.save(audio_file)

    print("\nVoice response generated!")

    display(Audio(audio_file, autoplay=False))

    return ai_response, audio_file

In [ ]:
ai_response, audio_file = voice_assistant(audio_path)